# Credit Card Fraud Detection - Model Evaluation

This notebook evaluates the performance of the trained XGBoost model on the credit card fraud dataset. We will calculate key performance metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC) and plot the Confusion Matrix, ROC Curve, and Precision-Recall Curve.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve, auc
)

# Set plotting style
sns.set_theme(style="whitegrid")

### 1. Load Data and Model

In [ ]:
# Load test dataset
# Note: test_dataset.csv contains a sample of 50 frauds and 950 legit transactions
df = pd.read_csv('../test_dataset.csv')
y_true = df['Class']

# Preprocess data using our pipeline
import sys
sys.path.append('..')
from src.preprocess import preprocess_data
X = preprocess_data(df.copy(), fit_scaler=False, scaler_path='../scaler.pkl')
if 'Class' in X.columns:
    X = X.drop(columns=['Class'])

# Load Model
model = joblib.load('../model.pkl')

# Generate Predictions
y_pred = model.predict(X)
y_prob = model.predict_proba(X)[:, 1]

### 2. Calculate Performance Metrics

In [ ]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

### 3. Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### 4. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 5. Precision-Recall Curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_true, y_prob)
pr_auc = auc(recall, precision)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.4f})', color='green', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()